# Many Models Forecasting Demo

This notebook showcases how to run MMF with MLForecast-based global models on multiple time series of hourly resolution. We use [`MLForecastLGBM`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py) (LightGBM with fixed hyperparameters) and [`MLForecastAutoLGBM`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/mlforecast/MLForecastPipeline.py) (Optuna-tuned LightGBM + feature pipeline), both built on top of [MLForecast](https://nixtlaverse.nixtla.io/mlforecast/index.html). We use [M4 competition](https://www.sciencedirect.com/science/article/pii/S0169207019301128#sec5) hourly data.

Unlike the [`global_hourly_dl.ipynb`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/hourly/global_hourly_dl.ipynb) notebook (GPU-bound neural forecasters), this notebook stays on CPU and runs end-to-end in a single Python process — no `dbutils.notebook.run` iteration needed.

### Cluster setup

We recommend using a **single-node CPU cluster** with [Databricks Runtime 18 for ML](https://docs.databricks.com/aws/en/release-notes/runtime/18ml). The pinned versions in [`requirements-global.txt`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/requirements-global.txt) (`mlforecast==1.0.31`, `lightgbm==4.6.0`, `optuna==3.6.1`) match the DBR 18 ML preinstalled versions exactly.

### Install and import packages
Check out [requirements-global.txt](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/requirements-global.txt) if you're interested in the libraries we use.

In [ ]:
%pip install -r ../../requirements-global.txt --quiet
%pip install datasetsforecast==0.0.8 --quiet
dbutils.library.restartPython()

In [ ]:
import logging
import pathlib
import pandas as pd
from datasetsforecast.m4 import M4
from mmf_sa import run_forecast

logging.getLogger("py4j.clientserver").setLevel(logging.WARNING)
logging.getLogger("py4j.java_gateway").setLevel(logging.WARNING)

### Prepare data
We are using [`datasetsforecast`](https://github.com/Nixtla/datasetsforecast/tree/main/) package to download M4 hourly data.

In [ ]:
# Number of time series
n = 100


def create_m4_hourly():
    y_df, _, _ = M4.load(directory=str(pathlib.Path.home()), group="Hourly")
    target_ids = {f"H{i}" for i in range(1, n)}
    y_df = y_df[y_df["unique_id"].isin(target_ids)]
    y_df = (
        y_df.groupby("unique_id", group_keys=False)
             .apply(lambda g: transform_group(g, g.name))
             .reset_index(drop=True)
    )
    return y_df


def transform_group(df, unique_id):
    if len(df) > 720:
        df = df.iloc[-720:]
    start = pd.Timestamp("2025-01-01 00:00")
    date_idx = pd.date_range(start=start, periods=len(df), freq="h", name="ds")
    res_df = pd.DataFrame({
        "ds": date_idx,
        "unique_id": unique_id,
        "y": df["y"].to_numpy()
    })
    return res_df

In [ ]:
catalog = "mmf"  # Name of the catalog we use to manage our assets
db = "m4"  # Name of the schema we use to manage our assets (e.g. datasets)
user = spark.sql('select current_user() as user').collect()[0]['user']  # User email address

In [ ]:
# Making sure that the catalog and the schema exist
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")

(
    spark.createDataFrame(create_m4_hourly())
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{catalog}.{db}.m4_hourly_train")
)

In [ ]:
display(
  spark.sql(f"select * from {catalog}.{db}.m4_hourly_train where unique_id in ('H1', 'H2', 'H3', 'H4', 'H5') order by unique_id, ds")
  )

### Models

Both models are configured in [`mmf_sa/models/models_conf.yaml`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/models/models_conf.yaml) with hourly-frequency-specific defaults in [`mmf_sa/forecasting_conf_hourly.yaml`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/mmf_sa/forecasting_conf_hourly.yaml). The hourly defaults use lags `[1, 24, 48, 168]` (yesterday, day-before, two-days, last-week-same-hour) and date features `hour` + `dayofweek`.

In [ ]:
active_models = [
    "MLForecastLGBM",
    "MLForecastAutoLGBM",
]

### Run MMF

Set `accelerator="cpu"` to dispatch to the CPU global-model path. We pass both models to `active_models` in a single call — LightGBM has no GPU-memory accumulation issue.

In [ ]:
run_forecast(
    spark=spark,
    train_data=f"{catalog}.{db}.m4_hourly_train",
    scoring_data=f"{catalog}.{db}.m4_hourly_train",
    scoring_output=f"{catalog}.{db}.hourly_scoring_output",
    evaluation_output=f"{catalog}.{db}.hourly_evaluation_output",
    model_output=f"{catalog}.{db}",
    group_id="unique_id",
    date_col="ds",
    target="y",
    freq="H",
    prediction_length=24,
    backtest_length=168,
    stride=24,
    metric="smape",
    train_predict_ratio=1,
    data_quality_check=True,
    resample=False,
    active_models=active_models,
    experiment_path=f"/Users/{user}/mmf/m4_hourly",
    use_case_name="m4_hourly",
    accelerator="cpu",
)

### Evaluate

In [ ]:
display(spark.sql(f"""
    select * from {catalog}.{db}.hourly_evaluation_output
    where unique_id = 'H1' and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by unique_id, model, backtest_window_start_date
    """))

### Forecast

In [ ]:
display(spark.sql(f"""
    select * from {catalog}.{db}.hourly_scoring_output
    where unique_id = 'H1' and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by unique_id, model, ds
    """))

Refer to the [post-evaluation analysis notebook](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/post-evaluation-analysis.ipynb).

### Delete Tables

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.hourly_evaluation_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.hourly_scoring_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))